# go-evo — Phase 1: random MLP genomes playing Go

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrimozRavbar/go-evo/blob/main/notebooks/phase1_colab.ipynb)

* **Go API:** [PettingZoo `classic.go_v5`](https://pettingzoo.farama.org/environments/classic/go/) (minigo engine). If it can't be installed/imported, the project falls back to its own engine (`goevo.engine.NativeGo`); tests check that the two engines agree on every move.
* **Players:** each genome is the flat weight vector of a **10-layer MLP** (PyTorch).
* **Phase 1:** 10 random genomes, each plays 5 headless games (25 games in total, distinct opponents, colours balanced). Then you can watch any game.

In [ ]:
# --- Setup (Colab clones the repo; locally it uses the checkout) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"   # change if the code lives on another branch, e.g. "claude/trusting-keller-qlgif5"
if IN_COLAB:
    if not os.path.exists("go-evo"):
        !git clone -q -b {BRANCH} https://github.com/PrimozRavbar/go-evo.git
    %cd go-evo
    !pip install -q -r requirements.txt
else:
    sys.path.insert(0, os.path.abspath(".."))  # running from notebooks/

In [ ]:
import numpy as np, torch
import goevo
from goevo import NetConfig, GameConfig, random_population, run_tournament, pettingzoo_available
from goevo.viz import animate_game, slider_viewer, watch_live, draw_board, ascii_replay

BACKEND = "pettingzoo" if pettingzoo_available() else "native"   # or force "native"
DEVICE = "cpu"  # tiny batch-1 nets: CPU is faster than GPU here
print("torch", torch.__version__, "| Go backend:", BACKEND)

## Configuration

In [ ]:
POP_SIZE        = 10
GAMES_PER_PLAYER = 5
SEED            = 0

net_cfg  = NetConfig(board_size=19, num_layers=10, hidden=128, activation="tanh")
game_cfg = GameConfig(board_size=19, komi=7.5, backend=BACKEND,
                      temperature=0.0,            # 0 = deterministic argmax over legal moves
                      forbid_own_eye_fill=True,   # don't let players fill their own eyes
                      min_moves_before_pass=None, # default N*N//2: no early resignation-by-passing
                      max_moves=None)             # default 3*N*N hard cap
print(f"MLP: {net_cfg.num_layers} Linear layers, dims {net_cfg.layer_dims()[0]} ... {net_cfg.layer_dims()[-1]}")
print(f"genome size: {net_cfg.genome_size:,} parameters")

## Create 10 random genomes and run the headless tournament

In [ ]:
players = random_population(POP_SIZE, net_cfg, seed=SEED)
result = run_tournament(players, GAMES_PER_PLAYER, game_cfg, seed=SEED)
print()
print(result.table())
print(f"\n{len(result.records)} games in {result.seconds:.1f}s")

In [ ]:
import matplotlib.pyplot as plt
order = np.argsort(-result.fitnesses())
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].bar([players[i].name for i in order], result.fitnesses()[order], color="#4472c4")
axs[0].set_title("fitness (wins + 0.5·draws, margin tiebreak)")
axs[1].hist([len(r.moves) for r in result.records], bins=10, color="#999")
axs[1].set_title("game length (moves)")
plt.tight_layout(); plt.show()

## Watch a game

Three options:
1. **Animation** of a recorded tournament game (play / pause / step buttons).
2. **Slider** to scrub through a recorded game.
3. **Live**: play a brand new game and watch it move by move.

In [ ]:
GAME_INDEX = 0   # 0 .. len(result.records)-1
rec = result.records[GAME_INDEX]
print(rec.black, "(B) vs", rec.white, "(W):", rec.result_string, "|", len(rec.moves), "moves")
animate_game(rec, interval_ms=250)

In [ ]:
slider_viewer(result.records[GAME_INDEX]);

In [ ]:
# Live game between the two best players of the tournament
best, second = players[order[0]], players[order[1]]
live_rec = watch_live(best, second, game_cfg, delay=0.15)

In [ ]:
# Plain-text replay (works anywhere); every=10 prints every 10th position
ascii_replay(result.records[GAME_INDEX], every=20)

## Save results

In [ ]:
from goevo import save_population
os.makedirs("results", exist_ok=True)
save_population("results/population.pt", [p.genome for p in players], net_cfg,
                [p.name for p in players], {"fitness": result.fitnesses().tolist()})
for k, r in enumerate(result.records):
    with open(f"results/game_{k:02d}.sgf", "w") as f:
        f.write(r.to_sgf())
print("saved", len(result.records), "SGF files + population.pt to results/")